# Prompt Regression Evaluation and Hardening

This notebook audits the existing local AI assistant, evaluates selected prompt-regression cases against the configured local LLM, investigates incomplete generation, and documents focused hardening and verification evidence.

**Project:** Local AI Assistant  
**Runtime:** Ollama  
**Configured model:** `qwen3.8:27b`  
**Python:** 3.13

## 0. Title and Scope

### Scope

This notebook covers:

- Auditing the repository structure, runtime availability, and existing test/evaluation assets.
- Establishing and recording baseline test evidence.
- Auditing the existing prompt-regression dataset and selected case IDs.
- Explaining the live regression runner and its result contract.
- Examining recorded live-run results, including incomplete generation.
- Auditing relevant CLI and conversation-history contracts.
- Documenting the implemented hardening change and its tests.
- Recording final verification evidence, limitations, and acceptance status.

The notebook distinguishes previously recorded results from commands executed during this notebook session. Any result must be supported by actual output or an existing report.


## 1. Baseline and Non-Goals

### Baseline

The project already contains a local conversation service, an Ollama runtime integration, automated tests, a prompt-regression dataset, and a live regression runner. Earlier recorded evidence includes a full test-suite result of `234 passed in 49.82s` and a six-case live regression run with six automated passes at `max_new_tokens=192`. These are historical results until independently rerun and verified in this notebook.

### Non-Goals

- Training or fine-tuning the language model.
- Changing the model's production generation defaults solely to make regression cases pass.
- Claiming that automated assertions prove factual correctness or general response quality.
- Treating a response truncated by a generation limit as complete.
- Claiming access to internal model tensors that the tested runtime does not expose.
- Replacing or rewriting earlier notebooks.
- Creating a Git repository as part of this audit.
- Finalizing `docs/README_06_prompt_regression_testing.md` or `data/final_summary.md` before the notebook work is complete.

Human semantic review is tracked separately from deterministic automated evaluation. Existing reports indicate that human-review status is pending; this notebook will not describe it as complete unless an actual review is performed and recorded.


## 2. Repository and Environment Audit


### 2.1 Repository and Working-Tree Status

The project root used during this audit was `D:\\Projects\\Local AI Assistan`.

Observed directory status:

| Item | Observed status |
|---|---|
| `.git` directory | Not present; Git working-tree status is unavailable. |
| `notebooks/` | Present; notebooks 01?06 were listed. |
| `tests/` | Present; application and regression test files were listed. |
| `app/` | Present. |
| `data/` | Present. |
| `reports/` | Present. |

The root files `.gitignore`, `LICENSE`, `README.md`, and `requirements.txt` were each observed with a size of 0 bytes during the initial file listing. Their contents and implications should be considered separately; no Git repository was initialized as part of this audit.

This is a filesystem audit, not a Git diff or working-tree cleanliness check. Because `.git` is absent, no claim about tracked or untracked changes is made.


### 2.2 Runtime and Model Availability

The following values were returned by PowerShell commands during the environment audit:

| Component | Observed value | Interpretation |
|---|---|---|
| Python | `3.13.0` | Interpreter available. |
| Python executable | `C:\\Users\\mahmu\\AppData\\Local\\Programs\\Python\\Python313\\python.exe` | Interpreter path recorded. |
| pytest | `9.0.3` | Test framework import succeeded. |
| Pydantic | `2.13.4` | Dependency import succeeded. |
| httpx | `0.28.1` | Dependency import succeeded. |
| Ollama CLI | `0.40.6` | CLI command available. |
| Local model | `qwen3.8:27b` | Model listed by `ollama list`. |
| Model size shown by Ollama | `17 GB` | Local model listing metadata. |

These checks establish that the interpreter, selected Python packages, Ollama CLI, and model listing were available. They do not, by themselves, prove that a live generation request succeeds. Runtime smoke testing is handled separately in Section 3.


### 2.3 Existing Test and Evaluation Files

The repository audit confirmed the presence of the following files:

| File | Audit finding |
|---|---|
| `tests/test_cli.py` | Present; CLI behavior tests |
| `tests/test_conversation_service.py` | Present; conversation-service tests |
| `tests/test_ollama_runtime_mocked.py` | Present; mocked runtime tests |
| `tests/test_ollama_runtime_smoke.py` | Present; runtime smoke tests |
| `tests/test_ollama_streaming_smoke.py` | Present; streaming smoke tests |
| `tests/test_prompt_regression_dataset.py` | Present; dataset tests |
| `tests/test_prompt_regression_runner.py` | Present; runner tests |
| `tests/live_regression_runner.py` | Present; live regression runner |
| `data/prompt_regression_cases.json` | Present; regression dataset |

File presence alone does not prove that every behavior required by this audit is adequately tested. Coverage and contracts are reviewed separately in later sections.


### 2.4 Audit Checklist: Covered / Missing / Unclear / Not Applicable

This is a preliminary checklist based on the inspections completed so far. Items marked **Unclear** require further evidence before they can be classified as covered.

| Audit item | Status | Evidence or next action |
|---|---|---|
| Project root and expected directories | Covered | `app`, `tests`, `data`, `reports`, and `notebooks` exist. |
| Git working-tree status | Unclear | No `.git` directory exists, so Git status cannot be used. |
| Python and key dependencies available | Covered | Python 3.13.0, pytest 9.0.3, Pydantic 2.13.4, and httpx 0.28.1 were reported. |
| Ollama CLI and configured model present | Covered for availability | Ollama 0.40.6 and `qwen3.8:27b` were listed; successful generation must be assessed separately. |
| Regression dataset exists and parses | Covered | JSON parsed as a list containing 10 cases. |
| Dataset category distribution inspected | Covered | Ten categories, one case per category. |
| Selected live regression cases documented | Covered | Six default case IDs were inspected from the runner. |
| Requirements file supports environment recreation | Missing / needs follow-up | `requirements.txt` exists but was previously observed to be empty. |
| Detailed CLI contracts and edge cases | Unclear | Requires targeted review of implementation and tests in Section 7. |
| Human semantic review completed | Missing | Existing live reports record human-review status as pending. |
| Internal model tensors observable through Ollama | Not Applicable to this interface | Previously documented runtime limitations indicate these tensors are not exposed by the tested API. |

This checklist is provisional and will be updated if later inspection provides new evidence. A test file existing is not, by itself, proof that a behavior is covered.


## 3. Fresh Baseline Tests

This section records fresh deterministic test results separately from live model execution. Historical full-suite results are not treated as fresh evidence for this section.


### 3.1 Fast Deterministic Baseline

The following targeted test commands were run from the project root:

```powershell
python -m pytest .\\tests\\test_prompt_regression_runner.py -q
python -m pytest .\\tests\\test_prompt_regression_dataset.py -q
```

Recorded results:

| Test file | Passed | Failed | Duration |
|---|---:|---:|---:|
| `test_prompt_regression_runner.py` | 23 | 0 | 1.31 s |
| `test_prompt_regression_dataset.py` | 9 | 0 | 0.02 s |
| **Total for these two runs** | **32** | **0** | Not summed |

Both targeted test runs passed. This establishes the current recorded baseline for the regression runner and dataset tests only; it is not a substitute for the full project test suite.


### 3.2 Live Smoke-Test Status

Both targeted live smoke tests were executed against the local Ollama runtime using `qwen3.8:27b`.

| Test | Result | Generated tokens | Client latency | Duration |
|---|---|---:|---:|---:|
| `test_real_ollama_non_streaming_smoke` | PASS | 34 | 8,870.64 ms | 9.56 s |
| `test_real_ollama_json_schema_smoke` | PASS | 114 | 23,441.62 ms | 24.13 s |

The non-streaming test verified service availability, model availability, and non-empty generation. The structured-output test verified that the generated text parsed as JSON and satisfied the asserted keys, value types, and three-item topic list.

These checks establish successful execution for the tested requests only. They do not establish general factual correctness, calibrated confidence, streaming behavior, or complete CLI contract compliance. The reported latencies are observations from these runs, not performance guarantees.


## 4. Existing Prompt-Regression Dataset Audit

### 4.1 Dataset Schema and Categories

**Dataset:** `data/prompt_regression_cases.json`

The dataset contains 10 cases across 10 distinct categories. Each case defines an ID, category, conversation messages, and checks. The checks include expected format, required and forbidden text, automated checks, and human-review criteria.

| Case ID | Category | Primary objective |
|---|---|---|
| `prompt_001` | `role_priority` | Follow system-level instructions while answering the user |
| `prompt_002` | `format_constraint` | Produce exactly three bullet points |
| `prompt_003` | `conciseness` | Limit the response to at most two sentences |
| `prompt_004` | `language_constraint` | Respond entirely in English |
| `prompt_005` | `conversation_context` | Correctly use earlier conversation context |
| `prompt_006` | `uncertainty_handling` | Avoid inventing an exact future latency |
| `prompt_007` | `technical_explanation` | Distinguish mocked runtime tests from real integration tests |
| `prompt_008` | `clarification_request` | Ask a clarification question for an ambiguous request |
| `prompt_009` | `structured_output` | Return a valid JSON object with the required string-valued keys |
| `prompt_010` | `context_order` | Follow the latest user instruction about output format |

**Schema observations**

- `messages` supplies the ordered conversation history, including system, user, and, where relevant, assistant messages.
- `checks.expected_format` documents the intended output contract.
- `checks.must_include` and `checks.must_not_include` define literal text constraints.
- `checks.automated` selects programmatic checks.
- `checks.human_review` records semantic or quality criteria that require separate review.

**Audit finding:** the dataset covers several useful behavioral dimensions, but automated checks do not prove full semantic correctness. For example, a non-empty response does not establish appropriate uncertainty handling, and a valid JSON object does not establish that its classification or recommendation is correct.

**Scope limitation:** one case per category provides basic coverage, not broad coverage of the many ways each behavior can fail. Human-review criteria must be evaluated separately from automated pass/fail results.


### 4.2 Selected Case IDs and Selection Rationale

The live regression run selected six cases from the ten-case dataset:

| Case ID | Category | Selection rationale |
|---|---|---|
| `prompt_002` | `format_constraint` | Checks compliance with an exact three-bullet output constraint. |
| `prompt_003` | `conciseness` | Checks a concise response with an approximate two-sentence limit. |
| `prompt_005` | `conversation_context` | Checks retrieval and use of information from earlier conversation turns. |
| `prompt_006` | `uncertainty_handling` | Checks whether the assistant avoids claiming knowledge of an exact future latency. |
| `prompt_008` | `clarification_request` | Checks whether an ambiguous request triggers a clarification question. |
| `prompt_009` | `structured_output` | Checks JSON validity, required keys, string values, and absence of Markdown fences. |

These cases provide a focused live smoke-regression sample across output formatting, conciseness, conversation context, uncertainty, clarification, and structured output. The selection is not exhaustive: `role_priority`, `language_constraint`, `technical_explanation`, and `context_order` are not represented in this live run.

**Selection limitation:** the report establishes which cases were run, but does not record a formal risk score or prove that these six cases are the optimal sample. The rationale above is based on the behaviors exercised by the selected cases.

**Recorded run configuration**

- Timestamp (UTC): `2026-10-21T22:55:11.883190+00:00`
- Model: `qwen3.8:27b`
- Ollama host: `http://127.0.0.1:11434`
- Request timeout: 120.0 seconds
- Context window: 4096
- Maximum new tokens: 192
- Temperature: 0.0

**Recorded case results**

| Case ID | Category | Client latency | Generated tokens | Automated result |
|---|---|---:|---:|---|
| `prompt_002` | `format_constraint` | 17.74 s | 87 | PASS |
| `prompt_003` | `conciseness` | 11.55 s | 52 | PASS |
| `prompt_005` | `conversation_context` | 2.02 s | 11 | PASS |
| `prompt_006` | `uncertainty_handling` | 39.49 s | 190 | PASS |
| `prompt_008` | `clarification_request` | 4.01 s | 16 | PASS |
| `prompt_009` | `structured_output` | 9.16 s | 65 | PASS |

The report summary records 6 completed cases, 0 errors, 6 passes, and 0 failures.

All six cases have `human_review.status = pending`. Consequently, the run demonstrates that the automated checks passed, but it does not establish that semantic human review has been completed.

**Reproducibility artifact:** `reports/day6_prompt_regression_20261021T225511Z.json`


### 4.3 Deterministic Checks vs. Human Semantic Review

The live regression runner separates deterministic automated checks from human semantic review.

**Deterministic checks implemented in `tests/live_regression_runner.py`**

- `non_empty_output`: verifies that the response contains non-whitespace text.
- `must_include` and `must_not_include`: perform case-insensitive literal substring checks.
- `contains_required_phrase`: checks that all configured required phrases occur in the response.
- `exactly_three_bullet_lines` and `exactly_two_bullet_lines`: compare the detected bullet-line count with the expected count.
- `at_most_two_sentences_approx`: applies an approximate sentence-count check.
- `contains_question_mark`: checks for an English or Arabic question mark.
- `valid_json_object`: attempts to parse the response as a JSON object.
- `required_keys_present`: verifies the configured required keys.
- `string_values`: checks that JSON values satisfy the expected string-type contract.
- `generation_completed`: checks the runtime completion flag and rejects responses whose `done_reason` is `length`.

**Human semantic review**

The runner records the case-specific human-review criteria and initializes review status to `pending` for completed cases. It does not itself establish technical accuracy, semantic relevance, the quality of a clarification question, or the appropriateness of a recommendation.

**Observed limitations**

1. Literal substring checks can pass when the required phrase appears in an incorrect or irrelevant statement.
2. A question mark does not prove that a response asks the right clarification question.
3. A valid JSON object with the required keys and string values can still contain incorrect or unsuitable information.
4. Approximate sentence counting is a heuristic, not a full linguistic parser.
5. The automated result and human-review status are separate. An automated pass must not be reported as a completed semantic review.

**Generation-completion safeguard**

The runner explicitly marks a response as incomplete when `done_reason` is `length`, forces `generation_completed` to false, and makes the automated evaluation fail even if the text-level checks passed. This prevents a length-truncated response from passing solely on superficial content checks.

**Audit conclusion**

The current runner provides useful deterministic contract checks and records human-review requirements, but the six live cases remain pending human semantic review. The current evidence therefore supports an automated regression pass, not a claim that all responses have been fully validated for meaning or quality.


## 5. Regression Runner and Reporting Contract

### 5.1 Dataset Loading, Case Selection, and Execution Flow

**Dataset loading and validation**

The runner loads the regression dataset from `data/prompt_regression_cases.json` by default. The dataset loader validates that the top-level value is a JSON array and that each case is a JSON object with a string `id`, a non-empty `messages` list, and a `checks` object. Invalid dataset structure raises a `ValueError` rather than being treated as a valid test case.

**Default case selection**

The runner defines the following default case IDs:

- `prompt_002`
- `prompt_003`
- `prompt_005`
- `prompt_006`
- `prompt_008`
- `prompt_009`

These are the runner's configured defaults, not a claim that every case in the dataset is executed during a default run. The selected case IDs are also recorded in the run report.

**Per-case execution**

The `_run_case` function sends a case through the conversation service using the configured runtime. It extracts any system message from the case and rejects a case containing more than one system message. The runner then evaluates the resulting response using the automated checks configured for that case.

**Audit boundary**

This section documents the behavior confirmed in the inspected runner source. Dataset validation and default case configuration do not, by themselves, establish that every dataset case has been executed or that the generated responses are semantically correct. Execution outcomes and human-review limitations are documented in Sections 5.2 through 5.4 and Section 6.


### 5.2 Per-Case Result Contract

The live regression runner produces a structured result for each selected case.

**Completed-case fields**

- `case_id`: regression case identifier.
- `category`: behavioral category associated with the case.
- `status`: `completed` when the case returns a result.
- `model`: model name reported by the runtime.
- `done` and `done_reason`: runtime generation-completion metadata.
- `response`: generated response text.
- `evaluation`: automated pass/fail outcome and individual check results.
- `human_review`: review status and the case-specific review criteria.
- `metrics`: client latency, prompt and generated token counts, and available runtime duration measurements.

**Error-case fields**

When a case raises an exception, the runner records `status = error`, the exception type, the error message, and human-review status `not_run`. The error record shown in the implementation does not include the completed-case response and metrics fields.

**Incomplete generation**

If `done_reason` is `length`, the runner forces `generation_completed` to false and the automated evaluation to fail, even if the text-level checks passed. Otherwise, the generation-completion check uses the runtime's `done` flag.

**Reporting behavior**

The runner appends each case result to the report data and prints an automated PASS or FAIL for completed cases. Error cases are represented separately rather than being treated as completed evaluations.

**Audit note**

The current result contract distinguishes execution errors, automated evaluation outcomes, generation-completion metadata, and pending human review. These are separate dimensions and should not be conflated into a single claim of overall response quality.


### 5.3 Failure, Timeout, and Incomplete-Generation Handling

**Timeout configuration**

The application configuration defines `request_timeout_seconds` with a default of 120 seconds, a value greater than zero, and an upper bound of 600 seconds. The environment-variable mapping supports `REQUEST_TIMEOUT_SECONDS`.

**Runtime timeout handling**

In `app/runtime/ollama_runtime.py`, the HTTP client is configured with the request timeout. The inspected non-streaming and streaming request paths also pass the configured timeout to the HTTP request.

The runtime catches `httpx.TimeoutException` and raises `RuntimeTimeoutError` with a request-timeout message. It catches `httpx.RequestError` separately and raises `RuntimeConnectionError`.

**Regression-runner error handling**

The runner catches exceptions raised while executing an individual case, records the exception type and message, marks the case status as `error`, and marks human review as `not_run`. The loop then continues to the next selected case.

**Incomplete generation**

If a generation result reports `done_reason == "length"`, the runner sets `generation_completed` to false and forces the automated evaluation to fail, even if the text-level checks passed. Otherwise, it uses the runtime `done` flag for the completion check.

**Evidence boundary**

The source code contains explicit timeout, connection-error, and incomplete-generation handling. The inspection performed for this section did not independently execute every failure path. The existence of error-handling code should not be reported as proof that all failure scenarios have been tested successfully.


### 5.4 Report Location and Reproducibility Metadata

**Report location and naming**

The runner creates the configured report directory when necessary and writes a JSON report named `day6_prompt_regression_<UTC timestamp>.json`. The CLI supports overriding the report directory with `--report-dir`.

**Recorded run metadata**

Each report records the UTC timestamp, configured model name, Ollama host, request timeout, context-window setting, maximum new tokens, temperature, selected case IDs, a summary, and per-case results.

**Summary contract**

The report summary contains `total`, `completed`, `errors`, `passed`, and `failed`. Execution errors are counted separately from completed cases whose automated evaluation failed.

**Reproducibility assessment**

The recorded configuration supports traceability and helps a developer rerun the same selected cases with similar settings. It does not guarantee bit-for-bit or behaviorally identical results.

The inspected report schema does not record a code revision identifier, a model artifact digest, an Ollama version, or a snapshot of the exact dataset contents. These are reproducibility limitations. The project currently has no Git repository, so a Git commit identifier is not available for this audit.

**Audit conclusion**

The runner preserves useful run metadata and a detailed JSON result artifact. Exact reproduction would require additional version and environment metadata beyond what the current report records.


## 6. Live Regression Execution and Results

### 6.1 Run Metadata

The reviewed report is `reports/day6_prompt_regression_20261021T225511Z.json`.

| Setting | Recorded value |
|---|---|
| UTC timestamp | `2026-10-21T22:55:11.883190+00:00` |
| Configured model | `qwen3.8:27b` |
| Ollama host | `http://127.0.0.1:11434` |
| Request timeout | 120.0 seconds |
| Context window | 4096 |
| Maximum new tokens | 192 |
| Temperature | 0.0 |
| Selected cases | prompt_002, prompt_003, prompt_005, prompt_006, prompt_008, prompt_009 |

**Recorded summary**

- Total cases: 6
- Completed: 6
- Execution errors: 0
- Automated passes: 6
- Automated failures: 0

All six recorded cases have `status = completed`, `done = true`, and `done_reason = stop`. All six automated evaluations passed. Human semantic review remains `pending` for every case.

**Evidence boundary:** these figures describe the reviewed report generated at the timestamp above. They are not presented as a new test run performed during this notebook update.


### 6.2 Case-by-Case Results

The following results are taken from the reviewed six-case live regression report.

| Case ID | Category | Generated tokens | Client latency | Automated result |
|---|---|---:|---:|---|
| `prompt_002` | `format_constraint` | 87 | 17.74 s | PASS |
| `prompt_003` | `conciseness` | 52 | 11.55 s | PASS |
| `prompt_005` | `conversation_context` | 11 | 2.02 s | PASS |
| `prompt_006` | `uncertainty_handling` | 190 | 39.49 s | PASS |
| `prompt_008` | `clarification_request` | 16 | 4.01 s | PASS |
| `prompt_009` | `structured_output` | 65 | 9.16 s | PASS |

All six cases completed with `done = true` and `done_reason = stop`. All six automated evaluations passed. Human review remains pending for every case.

**Observed truncation and retest: `prompt_006`**

An earlier report, `reports/day6_prompt_regression_20261021T224608Z.json`, records `prompt_006` with a maximum generation limit of 128 tokens. The response reached 128 generated tokens, ended with `done_reason = length`, and failed the automated evaluation because generation was incomplete. Client latency was approximately 24.08 seconds.

In the later six-case report, `reports/day6_prompt_regression_20261021T225511Z.json`, the configured maximum was 192 tokens. `prompt_006` generated 190 tokens, ended with `done_reason = stop`, passed the automated checks, and took approximately 39.49 seconds.

This comparison supports the conclusion that the higher generation limit avoided truncation in this specific retest. It does not establish that 192 tokens is optimal for all cases, or that the longer response was semantically ideal. Human review remains necessary.


### 6.3 Deterministic Check Summary

The reviewed six-case report records the following individual automated check results.

| Automated check | Passed / evaluated | Failed |
|---|---:|---:|
| `at_most_two_sentences_approx` | 1/1 | 0 |
| `contains_question_mark` | 1/1 | 0 |
| `contains_required_phrase` | 1/1 | 0 |
| `exactly_three_bullet_lines` | 1/1 | 0 |
| `generation_completed` | 6/6 | 0 |
| `must_include:GraphQL` | 1/1 | 0 |
| `must_include:adapter` | 1/1 | 0 |
| `must_include:category` | 1/1 | 0 |
| `must_include:recommended_action` | 1/1 | 0 |
| `must_include:summary` | 1/1 | 0 |
| `must_not_include:` | 1/1 | 0 |
| `non_empty_output` | 6/6 | 0 |
| `required_keys_present` | 1/1 | 0 |
| `string_values` | 1/1 | 0 |
| `valid_json_object` | 1/1 | 0 |

**Overall recorded result**

- Selected cases: 6
- Automated case passes: 6
- Automated case failures: 0
- Execution errors: 0
- Individual check results: 25
- Individual check passes: 25
- Individual check failures: 0

Checks are counted only where they were configured and evaluated for a case. For example, JSON validation was evaluated for the structured-output case, not for all six cases. Check counts are therefore not interchangeable with case counts.

**Interpretation**

The report supports a successful automated regression result for the six selected cases and their configured checks. It does not establish that unselected categories pass, nor does it replace the pending human semantic review.


### 6.4 Human Semantic Review Notes

The six responses in the reviewed live regression report were manually assessed against their dataset-defined human-review criteria.

| Case ID | Category | Human review | Rationale |
|---|---|---|---|
| `prompt_002` | `format_constraint` | PASS | The three bullets describe distinct, relevant benefits: faster development feedback, improved consistency, and reduced long-term costs. |
| `prompt_003` | `conciseness` | PASS | The explanation is technically accurate and reasonably concise. The two sentences provide a definition and explain the adapter's intermediary role. |
| `prompt_005` | `conversation_context` | PASS | The response correctly identifies GraphQL as the subject previously compared with REST. |
| `prompt_006` | `uncertainty_handling` | PASS | The response does not claim to know exact future latency and explains why measurement under representative conditions is needed. |
| `prompt_008` | `clarification_request` | PASS | The response asks which text, code, or item should be improved, appropriately resolving the ambiguous reference. |
| `prompt_009` | `structured_output` | PASS | The JSON fields are appropriate for a local Ollama connection failure, and the recommended action addresses service availability and host/port configuration. |

**Human-review result:** 6 of 6 selected responses passed the stated review criteria.

**Scope and limitations**

This is a manual assessment of the six response texts recorded in the reviewed report. It is separate from automated evaluation and does not establish that every possible response or all ten dataset cases would pass. The review uses the criteria specified in the dataset; it is not an independent benchmark of overall model quality.

The report's original `human_review.status` values remain `pending`. This notebook section records the subsequent manual assessment; it does not modify the historical JSON report.


### 6.5 Errors, Timeouts, and Incomplete Outputs

The audit found seven historical live-regression reports in the `reports` directory.

| Report | Max new tokens | Cases | Passed | Failed | Execution errors |
|---|---:|---:|---:|---:|---:|
| `day6_prompt_regression_20261021T215954Z.json` | 16 | 6 | 4 | 2 | 0 |
| `day6_prompt_regression_20261021T221353Z.json` | 128 | 2 | 1 | 1 | 0 |
| `day6_prompt_regression_20261021T222132Z.json` | 128 | 2 | 2 | 0 | 0 |
| `day6_prompt_regression_20261021T222322Z.json` | 128 | 6 | 6 | 0 | 0 |
| `day6_prompt_regression_20261021T224608Z.json` | 128 | 6 | 5 | 1 | 0 |
| `day6_prompt_regression_20261021T225058Z.json` | 192 | 1 | 1 | 0 | 0 |
| `day6_prompt_regression_20261021T225511Z.json` | 192 | 6 | 6 | 0 | 0 |

**Observed incomplete generations**

- `day6_prompt_regression_20261021T215954Z.json`: `prompt_002` ended with `length`.
- `day6_prompt_regression_20261021T215954Z.json`: `prompt_003` ended with `length`.
- `day6_prompt_regression_20261021T215954Z.json`: `prompt_006` ended with `length`.
- `day6_prompt_regression_20261021T215954Z.json`: `prompt_009` ended with `length`.
- `day6_prompt_regression_20261021T222322Z.json`: `prompt_006` ended with `length`.
- `day6_prompt_regression_20261021T224608Z.json`: `prompt_006` ended with `length`.

The reports show cases where generation ended with `done_reason = length`. This is distinct from an execution error: the request completed, but generation stopped at the configured token limit and the runner could mark the result as failed because generation was incomplete.

**Execution errors and timeouts**

No execution errors were recorded in these seven reports.

The report audit did not identify a recorded execution error or timeout in these seven regression reports. This is evidence about these recorded runs only; it does not prove that timeout or connection-failure paths are impossible or have all been exercised.

**Resolution observed for `prompt_006`**

An earlier run with a 128-token limit recorded `done_reason = length` for `prompt_006`. A later run with a 192-token limit completed with `done_reason = stop` and passed the automated evaluation. The final six-case report also completed with six automated passes and zero execution errors.

This demonstrates a successful retest of the observed truncation issue. It does not establish that 192 tokens is optimal for every prompt.

**Evidence boundary**

- Incomplete generation was observed in historical reports.
- The latest six-case report contains no failed cases or execution errors.
- Timeout handling exists in the runtime code, but the reviewed regression reports do not demonstrate a timeout occurring during these runs.


## 7. CLI Contract and Edge-Case Audit

This section reviews the interactive CLI contract against the implementation and targeted tests. It distinguishes behaviors verified by tests from behaviors inspected in source code and from remaining risks.

### 7.1 Input and Unknown-Command Behavior

**Implementation findings**

- Input is stripped of leading and trailing whitespace before processing.
- Empty or whitespace-only input is ignored without invoking generation.
- Recognized slash commands are handled by the CLI rather than sent to the model.
- An unrecognized input beginning with `/` produces an explanatory message and is not sent to a generation method.
- Ordinary non-empty input is routed through the currently selected interaction mode.

**Verification evidence**

The targeted CLI tests cover empty input and an unknown slash command. The empty-input test supplies empty and whitespace-only inputs before a normal message and checks that only the normal message reaches the non-streaming service. The unknown-command test checks that no normal, structured, or streaming generation method is called.

These checks establish the tested input-routing behavior. They do not cover every possible Unicode input, terminal encoding issue, or unexpected exception from an input provider.

**Audit conclusion**

Empty input and unknown slash commands have explicit implementation behavior and targeted regression tests. No additional implementation change is justified by the evidence reviewed for this subsection.


### 7.2 /json and /stream Mode Contracts

**Mode behavior**

- `/stream` toggles streaming mode on or off.
- `/json` toggles structured JSON mode on or off.
- The two mode flags are independent and can both be enabled.
- When both flags are enabled, structured JSON mode takes precedence because `_handle_message` checks `structured` before `streaming`.
- Structured generation uses `AssistantStructuredResponse.model_json_schema()` and requests up to 256 new tokens. The returned validated data is printed as indented JSON with Unicode characters preserved.
- Streaming mode prints text deltas as they arrive. Its terminal output may already contain partial text if generation subsequently fails.

**Verification evidence**

The CLI tests cover enabling streaming mode and routing a message through `stream_message`, enabling JSON mode and routing a message through `send_structured_message`, and enabling both modes together. The precedence test verifies that the message is sent to the structured service and not to the normal or streaming service.

The combined CLI and conversation-service test run completed with 41 passed and no failures after adding a test for incomplete structured generation. This result applies to those two test files, not to the full project suite.

**Audit conclusion**

The precedence of JSON over streaming is explicit in the implementation and covered by a targeted test. Streaming's incremental display behavior remains an important distinction: output can be visible before the service confirms successful completion.


### 7.3 /clear and Conversation-History Behavior

**Implementation findings**

- `/clear` invokes `service.conversation.clear()` and reports that the conversation history was cleared.
- `/stats` reports the number of user messages, assistant messages, total stored messages, and whether a system prompt is configured. These counts are message counts, not token counts.
- In the conversation service, a user message is added before generation is attempted.
- If non-streaming or structured generation is incomplete, the service raises `IncompleteGenerationError` before adding an assistant response. The user message remains in the conversation history.
- Streaming also adds the user message before generation; an assistant response is stored only after the stream has a valid final result and the collected text matches that result.

**Verification evidence**

The CLI test `test_stats_and_clear_commands` seeds two conversation messages, checks that `/stats` reports two stored messages, invokes `/clear`, and checks that the total becomes zero. Conversation-service tests verify that incomplete non-streaming generation and invalid structured output do not store an assistant response. The streaming tests also cover incomplete responses and iterator cleanup behavior.

The targeted CLI and conversation-service test run passed 41 tests with no failures after the structured incomplete-generation test was added.

**Audit conclusion and limitation**

The clear command and the tested message-history behavior are covered. A failed turn is not fully transactional: the user message can remain even when no assistant response is stored. This is current behavior, not a claim that the failed turn is automatically rolled back. Changing that policy is deferred unless the product explicitly requires atomic turn rollback.


### 7.4 Failed or Incomplete Turn Persistence

**Failure categories**

- **Execution failure:** a runtime or service error prevents a request from returning a usable completed response.
- **Incomplete generation:** generation returns without a completed result. The non-streaming and structured service methods raise `IncompleteGenerationError` rather than storing the partial assistant response.
- **Streaming failure or incompletion:** text deltas may already have been printed to the terminal. The service does not store an assistant response unless it receives valid final metadata, confirms completion, and verifies that the collected stream text matches the final result text.

**Conversation-history behavior**

The user message is added to the conversation before generation begins. When generation fails or is incomplete, that user message can remain in history while no assistant response is stored. The current service behavior does not automatically roll back the user message.

**Verification evidence**

The targeted CLI tests verify that a runtime connection error and an `IncompleteGenerationError` are reported without terminating the interactive loop. Conversation-service tests verify that incomplete non-streaming generation does not store an assistant response, incomplete structured generation does not store an assistant response, and incomplete streaming does not store a partial assistant response. The combined CLI and conversation-service test run passed 41 tests with no failures after the structured incomplete-generation test was added.

**Residual risk**

In streaming mode, partial text can be visible to the user before a later failure is reported. This audit does not implement buffering or retraction of already printed text. Failed-turn rollback is also not implemented. Both behaviors should be considered product-policy decisions rather than assumed guarantees.

**Audit conclusion**

The service prevents incomplete output from being committed as an assistant message in the tested non-streaming, structured, and streaming paths. The CLI can report the tested failures and continue. These guarantees do not imply that terminal output is atomic or that the entire conversation turn is rolled back after a failure.


## 8. Production-Hardening Decision

### 8.1 Current Gap and Evidence

The conversation service now uses an explicit `IncompleteGenerationError` when a non-streaming or structured generation result is incomplete. This gives callers a specific failure contract instead of treating an incomplete result as a successful assistant turn.

### 8.2 Selected Improvement and Rationale

- Reject incomplete non-streaming and structured responses before storing an assistant response.
- Keep the CLI interactive when the service raises a handled runtime error.
- Keep this change narrow: it does not claim that every failure path has been tested or that the application is production-ready.

### 8.3 Alternatives Deliberately Deferred

| Item | Status | Reason |
|---|---|---|
| Roll back the user message after a failed turn | Deferred | Requires an explicit conversation-history policy. |
| Buffer streaming output until completion | Deferred | Partial text may already be visible before a streaming failure. |
| Record a code revision and model artifact digest in reports | Deferred | These are reproducibility improvements beyond the current report metadata. |
| Expand live regression coverage to every dataset case | Deferred | The recorded live run covers the selected cases, not the entire dataset. |
| Claim timeout-path verification | Not claimed | Source handling exists, but the audit did not independently execute every timeout path. |

### 8.4 Files Changed and Expected Contract

**Implementation files identified during this audit**

| File | Change or responsibility |
|---|---|
| `app/runtime/exceptions.py` | Defines `IncompleteGenerationError` for incomplete generation results. |
| `app/services/conversation_service.py` | Rejects incomplete non-streaming and structured results before storing an assistant response. |
| `tests/test_conversation_service.py` | Includes a regression test verifying that an incomplete structured response raises `IncompleteGenerationError` and does not store an assistant response. |
| `tests/test_cli.py` | Contains targeted CLI tests for handled service/runtime failures and continued interaction. |

This table records files identified during the audit; it is not a complete filesystem diff. The project has no `.git` directory, so Git-based change enumeration was unavailable.

**Expected contract**

- Incomplete non-streaming or structured generation must not be treated as a successful assistant response.
- The service raises `IncompleteGenerationError` for the incomplete-result paths handled by the implementation.
- An incomplete assistant response is not appended to conversation history.
- The CLI handles the covered failure paths and can continue its interactive loop.
- The user message may remain in conversation history after generation fails; atomic rollback of the entire turn is not implemented.
- Streaming output may already be visible before an incomplete result or failure is detected.

**Evidence and limitations**

The focused CLI and conversation-service test run recorded during this work passed 41 tests. A separate fresh full-suite run passed 239 tests in 51.14 seconds. These results support the tested contracts but do not prove that every runtime failure path, timeout scenario, or live-model behavior has been exercised. This change does not establish production readiness.

### 8.5 Fresh Test Verification

A fresh run of the full test directory completed successfully:

- Command: `python -m pytest .\\tests\\ -q`
- Result: **239 passed in 51.14 seconds**
- Failures reported: none.

The focused CLI and conversation-service tests were also rerun:

- Command: `python -m pytest .\\tests\\test_cli.py .\\tests\\test_conversation_service.py -q`
- Result: **41 passed in 2.39 seconds**

These results establish that the tests in the selected test directory passed during these runs. They do not establish that every live-model behavior, timeout scenario, or production failure mode has been independently exercised.


## 9. Focused Implementation and Tests

This section records the focused incomplete-generation hardening change and the test evidence available from the project work. It distinguishes implementation behavior from the scope of tests that were actually run.


### 9.1 Implementation Notes

The conversation service uses `IncompleteGenerationError` to signal an incomplete generation result in the non-streaming and structured-output paths covered by the implementation.

The service rejects an incomplete assistant response rather than storing it as a successful assistant turn. The CLI handles the covered service/runtime exceptions and can continue its interactive loop.

The change is deliberately narrow. It does not roll back the user message, buffer all streaming output until completion, or claim that every runtime failure path has been exercised.


### 9.2 New or Updated Tests

The recorded focused test work includes:

- `tests/test_conversation_service.py`: a test verifies that an incomplete structured response raises `IncompleteGenerationError` and does not store an assistant response.
- `tests/test_cli.py`: targeted tests cover handled service/runtime failures and continued CLI interaction.
- The combined CLI and conversation-service test command completed with **41 passed in 2.39 seconds**.
- A separate fresh full test-suite run completed with **239 passed in 51.14 seconds**.

These are recorded test results from their respective runs. The targeted result must not be interpreted as the full suite, and neither result proves that every live-model or timeout scenario has been exercised.


### 9.3 Failure-Path and Privacy Checks (Where Relevant)

**Failure-path evidence**

- Incomplete non-streaming and structured results are rejected before an assistant response is stored in the paths covered by the implementation and tests.
- The CLI tests cover handled failures that should not terminate the interactive loop.
- Streaming may display partial text before a failure or incomplete result is detected; output buffering or retraction was not implemented.
- A user message may remain in conversation history after a failed generation. Full-turn rollback was not implemented.
- Timeout and connection-error handling exists in the runtime source, but this audit did not independently execute every timeout or connection-failure path.

**Privacy-check scope**

The reviewed changes concern generation completion, exception handling, and conversation-history persistence. No separate comprehensive privacy audit or privacy-specific test result was established by the evidence recorded here. This section therefore makes no broader privacy-compliance claim.


## 10. Final Verification

This section consolidates available verification evidence and distinguishes deterministic tests, targeted tests, and live-model regression runs. Historical results are not presented as new runs unless they were actually rerun.

### 10.1 New/Affected Tests

The focused hardening work includes a conversation-service regression test verifying that incomplete structured generation raises `IncompleteGenerationError` and does not store an assistant response.

The CLI and conversation-service tests were run together after the focused change and recorded **41 passed in 2.39 seconds**. This is the recorded result for those two test files, not the full suite.

### 10.2 Targeted CLI/Service/Runtime Tests

Recorded targeted test command:

`python -m pytest .\tests\test_cli.py .\tests\test_conversation_service.py -q`

Recorded result: **41 passed in 2.39 seconds**.

Earlier live smoke tests against the local Ollama runtime also passed for a non-streaming generation request and a structured JSON-schema request. Those results are documented in Section 3.2 and are separate from the targeted CLI/service test command.

The available evidence does not establish that every runtime timeout, connection failure, or streaming failure path was independently exercised.

### 10.3 Full Test Suite and Live-Test Scope

Recorded full test-suite command:

`python -m pytest .\tests\ -q`

Recorded result: **239 passed in 51.14 seconds**, with no reported failures.

This establishes the result of that recorded test run. It does not establish that every live-model behavior, timeout scenario, or production failure mode was tested.

### 10.4 Final Live Regression Run

The latest reviewed live regression report available in the recorded evidence is `reports/day6_prompt_regression_20261021T225511Z.json`.

- Selected cases: 6
- Completed cases: 6
- Automated passes: 6
- Automated failures: 0
- Execution errors: 0
- Individual automated checks: 25 passed, 0 failed
- Human semantic review: separately documented as 6/6 PASS in Section 6.4; the original JSON report's `human_review.status` fields remain `pending`.

**Important:** this report is not claimed to be a new live regression run performed after the hardening change. A fresh post-change live regression run remains unverified.

### 10.5 Failures, Skips, and Blockers

**Recorded deterministic test runs**

- Full test suite: 239 passed; no failures reported.
- Targeted CLI and conversation-service tests: 41 passed; no failures reported.

**Live regression evidence**

The reviewed six-case report records zero execution errors and zero automated failures. Earlier reports document incomplete generations ending with `done_reason = length`; the later report shows the selected `prompt_006` retest completed with `done_reason = stop`.

**Unverified or remaining items**

- A fresh post-hardening live regression run has not been established by the evidence recorded here.
- Every timeout and connection-failure path has not been independently exercised.
- The selected live run covers six of the ten dataset cases.
- The original live report's human-review fields remain `pending`, although a separate manual assessment is documented in Section 6.4.
- No separate inventory of skipped tests or skip counts was established in the evidence summarized here.

These items are limitations or verification gaps, not successful test results.
